# 3. Execute: expressions, calculations, constraints, requirements

The `Interpreter` evaluates KerML expressions, runs `calc` definitions as
functions, instantiates `part` definitions, and checks constraints and
requirements against the resulting instances.

In [ ]:
import longeron

model = longeron.loads("""
package Plant {
    part def Pump {
        attribute flowRate : Real = 40.0;      // L/min
        attribute power : Real = 1.5;          // kW
    }

    part def Plant {
        attribute demand : Real = 100.0;       // L/min
        part pumps : Pump[3];
        attribute capacity : Real = 3.0 * 40.0;
        assert constraint meetsDemand { capacity >= demand }
    }

    calc def EnergyCost {
        in kw : Real;
        in hours : Real;
        in rate : Real = 0.31;
        attribute kwh : Real = kw * hours;
        return : Real = kwh * rate;
    }

    requirement def DemandRequirement {
        subject plant : Plant;
        assume constraint { plant.demand > 0.0 }
        require constraint margin { plant.capacity >= 1.1 * plant.demand }
    }
}
""")
interp = longeron.Interpreter(model)

## Expression evaluation (arbitrary snippets, with bindings)

In [ ]:
print(interp.evaluate("2 ** 10 - 24"))
print(interp.evaluate("(1, 2, 3, 4)->select { in x; x % 2 == 0 }"))
print(interp.evaluate('if x > 0 ? "pos" else "neg"', x=-5))
print(interp.evaluate("sum(1..100)"))

## Calc definitions are callable (positional, named, defaults)

In [ ]:
print(interp.call("Plant::EnergyCost", 1.5, 24.0))
print(interp.call("Plant::EnergyCost", kw=4.5, hours=8.0, rate=0.25))

## Instantiation

Attribute defaults evaluate (they can reference siblings), nested parts
recurse, and exact multiplicities expand.

In [ ]:
plant = interp.instantiate("Plant::Plant")
print("capacity:", plant.slots["capacity"])
print("pumps:   ", len(plant.slots["pumps"]))
print(
    "pump 1:  ",
    plant.get("pumps_1.flowRate") if "pumps_1" in plant.slots else plant.slots["pumps"][0],
)
plant.to_dict()

## Constraint checking (and what-if overrides)

In [ ]:
for result in interp.check(plant):
    print(f"[{'PASS' if result.passed else 'FAIL'}] {result.name}: {result.expression}")

stressed = interp.instantiate("Plant::Plant", demand=150.0)
for result in interp.check(stressed):
    print(f"[{'PASS' if result.passed else 'FAIL'}] {result.name} with demand=150")

## Requirements: assumptions gate the verdict

In [ ]:
verdict = interp.check_requirement("Plant::DemandRequirement", subject=plant)
print("applicable:", verdict.applicable)
print("satisfied: ", verdict.satisfied)
for req in verdict.requirements:
    print(f"  require {req.name}: {req.passed}")

## The full loop: run, write results back, save

`Interpreter.snapshot` turns a runtime instance back into model elements
with bound values -- add it to a package and save in any format.

In [ ]:
import tempfile
from pathlib import Path

snapshot = interp.snapshot(plant, name="plantAsBuilt")
model.find("Plant").add(snapshot)

out = Path(tempfile.mkdtemp()) / "plant_with_results.sysml"
longeron.save(model, out)
print(out.read_text().split("part plantAsBuilt")[1][:200])